# About the Notebook

Moving from a diagnostic lab to a production API is where you move from **"agent vibes"**
to **"system physics."** This notebook implements the three operational pillars that keep
an agent alive in production:

1. **Checkpointing** — Periodic state snapshots so the agent can survive a restart.
2. **Context Pruning** — Token-aware history truncation to prevent "memory bloat."
3. **Circuit Breaker** — Turn-based and cost-based fuses to stop runaway loops.

| Pattern | Focus | Mechanism | The "Why" |
|---|---|---|---|
| Checkpointing | Reliability | Periodic state snapshots to Disk/Redis | If the process crashes at step 4 of 10, the agent resumes from step 4 instead of restarting. |
| Context Pruning | Performance | Token-aware history truncation | Prevents "Context Bloat" where old, irrelevant messages degrade reasoning and spike costs. |
| Circuit Breaker | Safety | Turn-based and cost-based fuses | Stops the agent if it enters an infinite loop of tool-call failures or exceeds a budget. |

We wire all three patterns together into a **hardened agent loop**, then run a live
showcase that proves each one works.

### 1. The `AgentSession` makes recovery possible
By making the agent's entire state serializable via Pydantic, you can snapshot it after
every turn. When the process crashes at step 4 of 10, you resume from step 4, not from
scratch. In production, this goes to Redis or Postgres, not a JSON file.

### 2. Context pruning is not optional
Models slow down and lose focus as the context grows. The Context Janitor keeps the system
prompt intact, preserves recent turns, and summarizes what was dropped. This isn't cleanup, it's a performance and cost control mechanism.

### 3. Hard-code a budget cap per session
The circuit breaker is your primary defense against infinite loops. An agent stuck in a
tool-call cycle will burn your API budget in minutes. A $0.50 cap per session is cheap
insurance. When the breaker trips, your FastAPI layer catches the exception and returns a
user-friendly "I need a human" message.

### 4. These patterns compose
Checkpointing, pruning, and the circuit breaker aren't separate features, they're layers of the same loop. Every turn: prune → call → update metrics → check breaker → checkpoint.
Remove any one and the system has a blind spot.

### 5. Streaming vs. Polling
In a FastAPI production setup, avoid standard REST `POST` requests for long-running agent
tasks. Agents take time to "think," and a 30-second timeout is common. Use **WebSockets**
or **Server-Sent Events (SSE)** to stream the agent's thought process to the UI. It doesn't
make the model faster, but seeing the agent "working" reduces perceived latency for the user.

In [ ]:
%pip install -q openai pydantic python-dotenv tiktoken

In [ ]:
import json, os, time, tempfile, shutil
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional
from dataclasses import dataclass, field

from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
BASE = "https://openrouter.ai/api/v1"

client = OpenAI(
    base_url=BASE,
    api_key=OPENROUTER_API_KEY,
)

MODEL = "openai/gpt-4o-mini"

# tiktoken encoding matched to the model — this gives us *exact* token
# counts for pre-call context management, not the chars÷4 guesswork
# you see in tutorials.
ENCODING = tiktoken.encoding_for_model("gpt-4o-mini")

print(f"Model:    {MODEL}")
print(f"API base: {BASE}")
print(f"Encoder:  {ENCODING.name} ({len(ENCODING._mergeable_ranks):,} merge rules)")

Model:    openai/gpt-4o-mini
API base: https://openrouter.ai/api/v1
Encoder:  o200k_base (199,998 merge rules)


## Agent Tools

In production, these functions call your order database, your policy service, and your
ticket classifier. Here we use deterministic stand-ins with realistic data so the
hardening patterns are testable and reproducible — but the tool schemas, the dispatcher,
and the function-calling contract are exactly what you would ship.

In [ ]:
# Tool definitions (OpenAI function-calling schema)

TOOL_SCHEMAS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "classify_ticket",
            "description": (
                "Classify a customer support ticket into a category. "
                "Returns the category and a confidence score."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {
                        "type": "string",
                        "description": "A one-sentence summary of the customer's issue.",
                    },
                    "category": {
                        "type": "string",
                        "enum": ["billing", "technical", "account", "general"],
                        "description": "The most fitting category for this ticket.",
                    },
                },
                "required": ["summary", "category"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": (
                "Look up an order by its ID and return current status, "
                "shipping estimate, and item details."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order identifier, e.g. ORD-1024.",
                    },
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_policy",
            "description": (
                "Look up a company policy by keyword. Useful for return, "
                "refund, warranty, or shipping policy questions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Policy topic to search for, e.g. 'return', 'refund', 'warranty'.",
                    },
                },
                "required": ["keyword"],
            },
        },
    },
]


# Mock data

_MOCK_ORDERS: dict[str, dict] = {
    "ORD-1024": {
        "status": "processing",
        "placed": "2025-02-04",
        "estimated_ship": "2025-02-14",
        "items": ["Wireless Keyboard x1", "USB-C Hub x1"],
    },
    "ORD-5567": {
        "status": "shipped",
        "placed": "2025-02-01",
        "tracking": "1Z999AA10123456784",
        "estimated_delivery": "2025-02-12",
        "items": ["Running Shoes (Size 10) x1"],
    },
    "ORD-8842": {
        "status": "delayed",
        "placed": "2025-01-20",
        "reason": "Supplier back-order; new estimated ship date 2025-02-18.",
        "items": ["Standing Desk Frame x1", "Monitor Arm x2"],
    },
}

_MOCK_POLICIES: dict[str, str] = {
    "return": (
        "Items may be returned within 30 days of delivery in original "
        "packaging. Electronics must be unopened or defective. Refunds "
        "are processed within 5-7 business days."
    ),
    "refund": (
        "Refunds are issued to the original payment method. Partial "
        "refunds may apply if the item shows signs of use. Shipping "
        "costs are non-refundable unless the return is due to our error."
    ),
    "warranty": (
        "All electronics carry a 1-year limited warranty covering "
        "manufacturing defects. Accessories are covered for 90 days. "
        "File a warranty claim through your account dashboard."
    ),
    "shipping": (
        "Standard shipping: 5-7 business days. Express: 2-3 business "
        "days. Free standard shipping on orders over $50. International "
        "shipping available to select countries."
    ),
}


# Mock tool implementations

def classify_ticket(summary: str, category: str) -> str:
    confidence = 0.92 if category in ("billing", "technical", "account") else 0.78
    return json.dumps({"category": category, "summary": summary, "confidence": confidence})


def lookup_order(order_id: str) -> str:
    order_id = order_id.strip().upper()
    if order_id in _MOCK_ORDERS:
        return json.dumps({"order_id": order_id, **_MOCK_ORDERS[order_id]})
    return json.dumps({"order_id": order_id, "error": "Order not found. Please verify the order ID."})


def check_policy(keyword: str) -> str:
    keyword = keyword.strip().lower()
    for key, text in _MOCK_POLICIES.items():
        if key in keyword or keyword in key:
            return json.dumps({"policy": key, "text": text})
    return json.dumps({"keyword": keyword, "text": "No specific policy found. Contact support@example.com."})


# Dispatcher

TOOL_MAP: dict[str, callable] = {
    "classify_ticket": classify_ticket,
    "lookup_order": lookup_order,
    "check_policy": check_policy,
}


def execute_tool(name: str, arguments: dict) -> str:
    """Dispatch a tool call by name. Returns a JSON string result."""
    fn = TOOL_MAP[name]
    return fn(**arguments)


print(f"✅ {len(TOOL_SCHEMAS)} tools registered: {list(TOOL_MAP.keys())}")
# Quick smoke test
print(f"   lookup_order('ORD-1024') → {lookup_order('ORD-1024')[:60]}...")
print(f"   check_policy('return')   → {check_policy('return')[:60]}...")

✅ 3 tools registered: ['classify_ticket', 'lookup_order', 'check_policy']
   lookup_order('ORD-1024') → {"order_id": "ORD-1024", "status": "processing", "placed": "...
   check_policy('return')   → {"policy": "return", "text": "Items may be returned within 3...


# The AgentSession: Serializable State

The `AgentSession` makes the agent's entire state serializable. This is the "brain" you
save to Redis, Postgres, or — in our showcase — a JSON file on disk.

Every field that matters for recovery is here: the conversation history, the turn counter,
the running cost, and the session status. If the process dies, you reload this object and
pick up exactly where you left off.

In [ ]:
class AgentSession(BaseModel):
    """Serializable agent state — the brain you checkpoint."""
    session_id: str
    model: str = MODEL
    system_prompt: str = "You are a customer support triage agent."
    history: List[Dict[str, Any]] = Field(default_factory=list)
    turn_count: int = 0
    total_tokens: int = 0
    total_usd: float = 0.0
    status: Literal["running", "halted", "completed"] = "running"
    halt_reason: Optional[str] = None
    checkpoints_saved: int = 0

    def add_message(self, role: str, content: str, **extra):
        """Append a message to the conversation history."""
        msg = {"role": role, "content": content, **extra}
        self.history.append(msg)

    def summary(self) -> str:
        return (
            f"Session {self.session_id} | status={self.status} | "
            f"turns={self.turn_count} | tokens={self.total_tokens} | "
            f"cost=${self.total_usd:.4f} | messages={len(self.history)} | "
            f"checkpoints={self.checkpoints_saved}"
        )


# Quick sanity check
s = AgentSession(session_id="test-001")
s.add_message("user", "Hello")
print(s.summary())
print(f"Serializes to {len(s.model_dump_json())} bytes of JSON")

Session test-001 | status=running | turns=0 | tokens=0 | cost=$0.0000 | messages=1 | checkpoints=0
Serializes to 267 bytes of JSON


# Token Counting & Cost Tracking

Two different jobs, two different sources of truth:

1. **Pre-call (context management):** Before we send a request, we need to know how
   many tokens the history will consume so the Context Janitor can prune. We use
   `tiktoken` — the same BPE tokenizer the model uses — to count exactly.
2. **Post-call (cost tracking):** After each API response, OpenRouter returns the
   actual `usage.prompt_tokens` and `usage.completion_tokens`. We use those for
   billing, not a local estimate.

In [ ]:
#  Pre-call: tiktoken-based token counting
def count_message_tokens(message: Dict[str, Any]) -> int:
    """Count tokens in a single chat message using tiktoken.

    Follows the OpenAI token-counting recipe: every message has a fixed
    overhead for role/name framing, plus the content tokens.
    See: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
    """
    # Per-message overhead (role, delimiters) — 4 tokens for gpt-4o/gpt-4o-mini
    tokens = 4
    for key, value in message.items():
        if isinstance(value, str):
            tokens += len(ENCODING.encode(value))
        elif isinstance(value, list):
            # tool_calls, repro_steps, etc.
            tokens += len(ENCODING.encode(json.dumps(value)))
    return tokens


def count_tokens(messages: List[Dict[str, Any]]) -> int:
    """Count total tokens for a full conversation history.

    Includes the 2-token reply primer that the API adds after the last message.
    """
    total = sum(count_message_tokens(m) for m in messages)
    total += 2  # reply primer
    return total


#  Post-call: cost from actual API-reported usage

# gpt-4o-mini pricing (per 1K tokens) — update these when pricing changes
COST_PER_1K_INPUT  = 0.00015   # $0.15  / 1M input tokens
COST_PER_1K_OUTPUT = 0.0006    # $0.60  / 1M output tokens


def calculate_cost(prompt_tokens: int, completion_tokens: int) -> float:
    """Calculate USD cost from the API-reported token counts."""
    return (
        (prompt_tokens / 1000) * COST_PER_1K_INPUT +
        (completion_tokens / 1000) * COST_PER_1K_OUTPUT
    )


#  Demo: show the difference between real counting and guessing

demo_msgs = [
    {"role": "system", "content": "You are a customer support triage agent."},
    {"role": "user", "content": "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on."},
]

real_count = count_tokens(demo_msgs)
naive_guess = sum(len(str(m.get("content", ""))) for m in demo_msgs) // 4

print(f"tiktoken count:  {real_count} tokens")
print(f"chars÷4 guess:   {naive_guess} tokens")
print(f"Error:           {abs(real_count - naive_guess)} tokens ({abs(real_count - naive_guess)/real_count*100:.0f}%)")
print(f"\nCost for 1000 in + 500 out: ${calculate_cost(1000, 500):.6f}")

tiktoken count:  41 tokens
chars÷4 guess:   30 tokens
Error:           11 tokens (27%)

Cost for 1000 in + 500 out: $0.000450


# The Context Janitor: History Pruning

High-end models have massive context windows, but filling them entirely is a mistake.
It increases latency and "smears" the agent's attention. The Janitor keeps the history
lean by:

1. **Never pruning the system prompt** — it contains the agent's core identity.
2. **Keeping only the most recent turns** — preserves immediate context.
3. **Injecting a summary** of what was pruned to maintain long-term memory.

In [ ]:
def prune_history(
    history: List[Dict],
    max_tokens: int = 4000,
    keep_recent: int = 6,
) -> List[Dict]:
    """
    The Context Janitor: keeps the system prompt and the last N turns,
    discarding the middle. Injects a summary of pruned content.

    Token counts are computed with tiktoken — the same BPE tokenizer
    the model uses — so the pruning threshold is exact, not a guess.
    """
    current_tokens = count_tokens(history)
    if current_tokens <= max_tokens:
        return history  # Nothing to prune

    # Never prune the system prompt (first message)
    system_msg = history[0] if history and history[0].get("role") == "system" else None

    if system_msg:
        middle = history[1:-keep_recent] if len(history) > keep_recent + 1 else []
        recent = history[-keep_recent:]
    else:
        middle = history[:-keep_recent] if len(history) > keep_recent else []
        recent = history[-keep_recent:]

    # Build a summary of what we're pruning
    n_pruned = len(middle)
    pruned_roles = [m.get("role", "?") for m in middle]
    role_counts = {r: pruned_roles.count(r) for r in set(pruned_roles)}
    role_summary = ", ".join(f"{count} {role}" for role, count in role_counts.items())

    summary_msg = {
        "role": "system",
        "content": (
            f"[Context Janitor] Pruned {n_pruned} older messages "
            f"({role_summary}) to stay within token budget. "
            f"Recent context preserved below."
        ),
    }

    pruned = []
    if system_msg:
        pruned.append(system_msg)
    pruned.append(summary_msg)
    pruned.extend(recent)

    after_tokens = count_tokens(pruned)
    print(
        f"  🧹 Context Janitor: {len(history)} msgs ({current_tokens:,} tokens) → "
        f"{len(pruned)} msgs ({after_tokens:,} tokens) | pruned {n_pruned} middle messages"
    )

    return pruned


# Demo: build a fat history and prune it
demo_history = [{"role": "system", "content": "You are a helpful agent."}]
for i in range(20):
    demo_history.append({"role": "user", "content": f"Turn {i}: " + "x" * 200})
    demo_history.append({"role": "assistant", "content": f"Reply {i}: " + "y" * 200})

print(f"Before: {len(demo_history)} messages, {count_tokens(demo_history):,} tokens (tiktoken)")
pruned = prune_history(demo_history, max_tokens=1500, keep_recent=6)
print(f"After:  {len(pruned)} messages, {count_tokens(pruned):,} tokens (tiktoken)")
print(f"First message role: {pruned[0]['role']}")
print(f"Second message: {pruned[1]['content'][:100]}...")

Before: 41 messages, 1,933 tokens (tiktoken)
  🧹 Context Janitor: 41 msgs (1,933 tokens) → 8 msgs (336 tokens) | pruned 34 middle messages
After:  8 messages, 336 tokens (tiktoken)
First message role: system
Second message: [Context Janitor] Pruned 34 older messages (17 assistant, 17 user) to stay within token budget. Rece...


# The Circuit Breaker

Hard-coding a budget cap per session is your primary defense against infinite loops.
The circuit breaker has two fuses:

- **Budget fuse**: If the session's cumulative cost exceeds `BUDGET_CAP`, halt immediately.
- **Turn fuse**: If the agent has taken more than `MAX_TURNS` reasoning steps, halt.

When the breaker trips, it raises a `CircuitBreakerTripped` exception that your FastAPI
layer can catch and turn into a user-friendly "I'm stuck" message.

In [ ]:
class CircuitBreakerTripped(RuntimeError):
    """Raised when the agent exceeds its operational budget or turn limit."""
    def __init__(self, reason: str, session: AgentSession):
        self.reason = reason
        self.session = session
        super().__init__(reason)


def check_circuit_breaker(
    session: AgentSession,
    budget_cap: float = 0.50,
    max_turns: int = 12,
):
    """
    Trip the fuse if the agent is burning money or stuck in a loop.
    Call this AFTER every LLM call.
    """
    if session.total_usd >= budget_cap:
        session.status = "halted"
        session.halt_reason = f"Budget exceeded: ${session.total_usd:.4f} >= ${budget_cap:.2f}"
        raise CircuitBreakerTripped(session.halt_reason, session)

    if session.turn_count >= max_turns:
        session.status = "halted"
        session.halt_reason = f"Max turns exceeded: {session.turn_count} >= {max_turns}"
        raise CircuitBreakerTripped(session.halt_reason, session)


# Demo: force a trip
demo_session = AgentSession(session_id="breaker-test", total_usd=0.48)
try:
    demo_session.total_usd += 0.05  # push over the $0.50 cap
    check_circuit_breaker(demo_session, budget_cap=0.50)
except CircuitBreakerTripped as e:
    print(f"⚡ Circuit breaker tripped: {e.reason}")
    print(f"   Session status: {e.session.status}")

⚡ Circuit breaker tripped: Budget exceeded: $0.5300 >= $0.50
   Session status: halted


# Checkpointing: Save & Restore

Checkpointing after every turn ensures that a 502 error or a container restart doesn't
force the user to start over. In production this goes to Redis or Postgres. For the
showcase we use a JSON file, which makes the checkpoint inspectable.

In [ ]:
CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix="agent_checkpoints_"))
print(f"Checkpoint directory: {CHECKPOINT_DIR}")


def save_checkpoint(session: AgentSession) -> Path:
    """Persist the full session state to disk."""
    path = CHECKPOINT_DIR / f"{session.session_id}.json"
    path.write_text(session.model_dump_json(indent=2))
    session.checkpoints_saved += 1
    return path


def load_checkpoint(session_id: str) -> Optional[AgentSession]:
    """Restore a session from its last checkpoint. Returns None if not found."""
    path = CHECKPOINT_DIR / f"{session_id}.json"
    if not path.exists():
        return None
    data = json.loads(path.read_text())
    return AgentSession.model_validate(data)


def list_checkpoints() -> List[str]:
    """List all saved session IDs."""
    return [p.stem for p in CHECKPOINT_DIR.glob("*.json")]


# Demo
demo = AgentSession(session_id="ckpt-demo", turn_count=3, total_usd=0.012)
demo.add_message("system", "You are a helpful agent.")
demo.add_message("user", "What is your return policy?")
demo.add_message("assistant", "Our return policy allows returns within 30 days.")

path = save_checkpoint(demo)
print(f"Saved checkpoint to: {path}")
print(f"Checkpoint size: {path.stat().st_size} bytes")

restored = load_checkpoint("ckpt-demo")
print(f"Restored: {restored.summary()}")
print(f"History intact: {len(restored.history)} messages")

Checkpoint directory: /tmp/agent_checkpoints_qu63q9s0
Saved checkpoint to: /tmp/agent_checkpoints_qu63q9s0/ckpt-demo.json
Checkpoint size: 551 bytes
Restored: Session ckpt-demo | status=running | turns=3 | tokens=0 | cost=$0.0120 | messages=3 | checkpoints=0
History intact: 3 messages


# The Hardened Agent Loop

This is where the three patterns come together. Every iteration of the loop:

1. **Prunes the context** before calling the model (Context Janitor)
2. **Calls the LLM** and executes any tool calls
3. **Updates metrics** (tokens, cost, turn count)
4. **Checks the circuit breaker** (budget + turns)
5. **Saves a checkpoint** (state persistence)

If any step fails, the last checkpoint lets you recover.

In [ ]:
MAX_TOOL_ROUNDS = 5  # Safety cap on tool-call loops within a single turn


def hardened_agent_step(
    session: AgentSession,
    user_input: str,
    budget_cap: float = 0.50,
    max_turns: int = 12,
    max_context_tokens: int = 4000,
) -> str:
    """
    Run one hardened agent turn: prune → call LLM → tools → metrics → breaker → checkpoint.
    Returns the assistant's final reply.
    """
    if session.status != "running":
        raise RuntimeError(f"Session {session.session_id} is {session.status}: {session.halt_reason}")

    # Ensure system prompt is in history
    if not session.history or session.history[0].get("role") != "system":
        session.history.insert(0, {"role": "system", "content": session.system_prompt})

    # Add the user message
    session.add_message("user", user_input)

    #  Context Janitor: prune before calling the model
    session.history = prune_history(
        session.history,
        max_tokens=max_context_tokens,
        keep_recent=6,
    )

    #  LLM call (with tool-call loop)
    final_reply = None
    for _round in range(MAX_TOOL_ROUNDS):
        response = client.chat.completions.create(
            model=session.model,
            messages=session.history,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0.2,
        )

        choice = response.choices[0]
        message = choice.message

        #  Update metrics
        usage = response.usage
        prompt_tok = (usage.prompt_tokens or 0) if usage else 0
        compl_tok = (usage.completion_tokens or 0) if usage else 0
        session.total_tokens += prompt_tok + compl_tok
        session.total_usd += calculate_cost(prompt_tok, compl_tok)

        # No tool calls → final answer
        tool_calls = getattr(message, "tool_calls", None)
        if not tool_calls:
            final_reply = message.content or "(no response)"
            session.add_message("assistant", final_reply)
            break

        # Execute tool calls
        session.history.append(message.model_dump())
        for tc in tool_calls:
            fn_name = tc.function.name
            try:
                fn_args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                fn_args = {}

            try:
                result_str = execute_tool(fn_name, fn_args)
            except Exception as exc:
                result_str = json.dumps({"error": str(exc)})

            session.history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result_str,
            })
            print(f"  🔧 Tool: {fn_name}({fn_args}) → {result_str[:80]}...")
    else:
        final_reply = "(Agent reached maximum tool-call rounds.)"
        session.add_message("assistant", final_reply)

    #  Increment turn counter
    session.turn_count += 1

    #  Circuit Breaker
    check_circuit_breaker(session, budget_cap=budget_cap, max_turns=max_turns)

    #  Checkpoint
    ckpt_path = save_checkpoint(session)
    print(f"  💾 Checkpoint saved (turn {session.turn_count}, ${session.total_usd:.4f})")

    return final_reply


print("Hardened agent loop ready.")

Hardened agent loop ready.


---

## Showcase: All Three Patterns in Action

Now we run the hardened agent through four scenarios that prove each pattern works:

1. **Normal multi-turn operation** — checkpointing after every turn
2. **Context pruning fires** — we bloat the history and watch the Janitor trim it
3. **Circuit breaker trips** — we set a low turn limit and watch the fuse blow
4. **Crash recovery** — we "kill" the session and restore from the last checkpoint

### Showcase 1: Normal Multi-Turn Operation

A realistic support conversation: the customer has a delayed order, asks about return
policy, and follows up. The agent uses tools, checkpoints every turn, and the circuit
breaker stays green.

In [ ]:
session1 = AgentSession(
    session_id="showcase-normal-001",
    model=MODEL,
    system_prompt=(
        "You are a customer support triage agent. Your job is to:\n"
        "1. Classify the ticket into a category (billing, technical, account, general).\n"
        "2. Assess priority (low, medium, high, urgent).\n"
        "3. Use available tools to gather context.\n"
        "4. Write a brief, empathetic response.\n"
        "Be concise. Never fabricate order details — use the lookup tool."
    ),
)

conversation = [
    "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on. This is unacceptable.",
    "OK, so it's delayed. What are my options? Can I return it when it arrives if it's too late?",
    "Fine. I'll wait, but if it's not here by the 18th I want a full refund. Please escalate this.",
]

print("=" * 70)
print(" SHOWCASE 1: Normal Multi-Turn Operation")
print("=" * 70)

for i, msg in enumerate(conversation, 1):
    print(f"\n{'─' * 70}")
    print(f"Turn {i} | Customer: {msg[:80]}...")
    print(f"{'─' * 70}")
    reply = hardened_agent_step(session1, msg)
    print(f"\n  🤖 Agent: {reply[:200]}..." if len(reply) > 200 else f"\n  🤖 Agent: {reply}")

session1.status = "completed"
save_checkpoint(session1)

print(f"\n{'=' * 70}")
print(f"Final: {session1.summary()}")
print(f"{'=' * 70}")

 SHOWCASE 1: Normal Multi-Turn Operation

──────────────────────────────────────────────────────────────────────
Turn 1 | Customer: I placed order #ORD-8842 three weeks ago and nobody has told me what's going on....
──────────────────────────────────────────────────────────────────────
  🔧 Tool: classify_ticket({'summary': "I placed order #ORD-8842 three weeks ago and nobody has told me what's going on.", 'category': 'general'}) → {"category": "general", "summary": "I placed order #ORD-8842 three weeks ago and...
  🔧 Tool: lookup_order({'order_id': 'ORD-8842'}) → {"order_id": "ORD-8842", "status": "delayed", "placed": "2025-01-20", "reason": ...
  💾 Checkpoint saved (turn 1, $0.0002)

  🤖 Agent: **Category:** General  
**Priority:** Urgent  

---

Dear Customer,

I sincerely apologize for the delay with your order #ORD-8842. It is currently delayed due to a supplier back-order, and the new es...

──────────────────────────────────────────────────────────────────────
Turn 2 | Customer: 

### Showcase 2: Context Pruning in Action

We deliberately bloat the history with 20 filler messages, then send a real question.
The Context Janitor should trim the fat before the LLM call, keeping the system prompt
and the most recent turns.

In [ ]:
session2 = AgentSession(
    session_id="showcase-pruning-002",
    model=MODEL,
    system_prompt="You are a customer support triage agent. Be concise.",
)

# Inject the system prompt
session2.history.append({"role": "system", "content": session2.system_prompt})

# Bloat the history with 20 turns of filler
for i in range(20):
    session2.history.append({"role": "user", "content": f"Filler question {i}: " + "blah " * 60})
    session2.history.append({"role": "assistant", "content": f"Filler answer {i}: " + "response " * 60})

print("=" * 70)
print(" SHOWCASE 2: Context Pruning in Action")
print("=" * 70)
print(f"\nHistory BEFORE: {len(session2.history)} messages, {count_tokens(session2.history):,} tokens")
print(f"\nSending a real question into the bloated context...\n")

reply = hardened_agent_step(
    session2,
    "Can you look up order #ORD-5567 for me?",
    max_context_tokens=1500,  # Force aggressive pruning
)

print(f"\nHistory AFTER: {len(session2.history)} messages, {count_tokens(session2.history):,} tokens")
print(f"\n  🤖 Agent: {reply[:300]}")
print(f"\n{session2.summary()}")

 SHOWCASE 2: Context Pruning in Action

History BEFORE: 41 messages, 2,899 tokens

Sending a real question into the bloated context...

  🧹 Context Janitor: 42 msgs (2,917 tokens) → 8 msgs (432 tokens) | pruned 35 middle messages
  🔧 Tool: lookup_order({'order_id': 'ORD-5567'}) → {"order_id": "ORD-5567", "status": "shipped", "placed": "2025-02-01", "tracking"...
  💾 Checkpoint saved (turn 1, $0.0003)

History AFTER: 11 messages, 670 tokens

  🤖 Agent: Order #ORD-5567 has been shipped. Here are the details:

- **Status:** Shipped
- **Placed on:** February 1, 2025
- **Tracking Number:** 1Z999AA10123456784
- **Estimated Delivery:** February 12, 2025
- **Items:** Running Shoes (Size 10) x1

Session showcase-pruning-002 | status=running | turns=1 | tokens=1403 | cost=$0.0003 | messages=11 | checkpoints=1


### Showcase 3: Circuit Breaker Trips

We set `max_turns=3` and send 5 messages. The breaker should trip on the 4th turn,
halting the session and raising `CircuitBreakerTripped`. In a FastAPI app, you'd catch
this and return a user-friendly "I need a human" message.

In [ ]:
session3 = AgentSession(
    session_id="showcase-breaker-003",
    model=MODEL,
    system_prompt="You are a customer support triage agent. Be concise.",
)

messages_to_send = [
    "What is your return policy?",
    "What about warranties?",
    "How long does shipping take?",
    "Can you look up order #ORD-1024?",  # This should trigger the breaker
    "One more question...",               # Should never reach this
]

print("=" * 70)
print(" SHOWCASE 3: Circuit Breaker Trips (max_turns=3)")
print("=" * 70)

for i, msg in enumerate(messages_to_send, 1):
    try:
        print(f"\n  Turn {i}: {msg}")
        reply = hardened_agent_step(session3, msg, max_turns=3)
        print(f"  🤖 {reply[:120]}..." if len(reply) > 120 else f"  🤖 {reply}")
    except CircuitBreakerTripped as e:
        print(f"\n  ⚡ CIRCUIT BREAKER TRIPPED on turn {i}!")
        print(f"     Reason: {e.reason}")
        print(f"     Session status: {e.session.status}")
        print(f"     → In production: return 'I need to hand you to a human agent.'")
        break
    except RuntimeError as e:
        print(f"\n  🚫 Session refused (already halted): {e}")
        break

print(f"\n{session3.summary()}")

 SHOWCASE 3: Circuit Breaker Trips (max_turns=3)

  Turn 1: What is your return policy?
  🔧 Tool: check_policy({'keyword': 'return'}) → {"policy": "return", "text": "Items may be returned within 30 days of delivery i...
  💾 Checkpoint saved (turn 1, $0.0001)
  🤖 Our return policy allows items to be returned within 30 days of delivery in their original packaging. Electronics must b...

  Turn 2: What about warranties?
  🔧 Tool: check_policy({'keyword': 'warranty'}) → {"policy": "warranty", "text": "All electronics carry a 1-year limited warranty ...
  💾 Checkpoint saved (turn 2, $0.0002)
  🤖 All electronics come with a 1-year limited warranty covering manufacturing defects, while accessories are covered for 90...

  Turn 3: How long does shipping take?
  🔧 Tool: check_policy({'keyword': 'shipping'}) → {"policy": "shipping", "text": "Standard shipping: 5-7 business days. Express: 2...

  ⚡ CIRCUIT BREAKER TRIPPED on turn 3!
     Reason: Max turns exceeded: 3 >= 3
     Session status: hal

### Showcase 4: Crash Recovery from Checkpoint

We simulate a crash by "forgetting" the session object, then restore it from the last
checkpoint. The agent picks up exactly where it left off. Same history, same metrics, same turn count.

In [ ]:
print("=" * 70)
print(" SHOWCASE 4: Crash Recovery from Checkpoint")
print("=" * 70)

# Step 1: Start a session and run 2 turns
session4 = AgentSession(
    session_id="showcase-recovery-004",
    model=MODEL,
    system_prompt="You are a customer support agent. Be concise and helpful.",
)

print("\n── Phase 1: Running 2 turns before the 'crash' ──")
reply1 = hardened_agent_step(session4, "I need to check on order #ORD-1024.")
print(f"  🤖 Turn 1: {reply1[:150]}")

reply2 = hardened_agent_step(session4, "When will it ship? I need it by Friday.")
print(f"  🤖 Turn 2: {reply2[:150]}")

print(f"\n  State before crash: {session4.summary()}")

# Step 2: Simulate the crash
print("\n── Phase 2: 💥 SIMULATED CRASH (deleting session object) ──")
crashed_session_id = session4.session_id
del session4  # Gone. Process died. Container restarted.
print(f"  Session object destroyed. Only the checkpoint remains.")

# Step 3: Recover from checkpoint
print("\n── Phase 3: Recovering from checkpoint ──")
print(f"  Available checkpoints: {list_checkpoints()}")

recovered = load_checkpoint(crashed_session_id)
if recovered:
    print(f"  ✅ Recovered: {recovered.summary()}")
    print(f"  History length: {len(recovered.history)} messages")
    print(f"  Last message: {recovered.history[-1]['content'][:100]}...")

    # Step 4: Continue the conversation
    print("\n── Phase 4: Continuing the conversation after recovery ──")
    reply3 = hardened_agent_step(recovered, "Actually, can you also check the shipping policy?")
    print(f"  🤖 Turn 3 (post-recovery): {reply3[:200]}")
    print(f"\n  Final state: {recovered.summary()}")
else:
    print(f"  ❌ No checkpoint found for {crashed_session_id}")

 SHOWCASE 4: Crash Recovery from Checkpoint

── Phase 1: Running 2 turns before the 'crash' ──
  🔧 Tool: lookup_order({'order_id': 'ORD-1024'}) → {"order_id": "ORD-1024", "status": "processing", "placed": "2025-02-04", "estima...
  💾 Checkpoint saved (turn 1, $0.0001)
  🤖 Turn 1: Your order #ORD-1024 is currently processing. It was placed on February 4, 2025, and is estimated to ship by February 14, 2025. The items in your orde
  💾 Checkpoint saved (turn 2, $0.0002)
  🤖 Turn 2: Your order is estimated to ship by February 14, 2025. If you need it by Friday, please note that it may not arrive in time, as February 14 is a Wednes

  State before crash: Session showcase-recovery-004 | status=running | turns=2 | tokens=1019 | cost=$0.0002 | messages=7 | checkpoints=2

── Phase 2: 💥 SIMULATED CRASH (deleting session object) ──
  Session object destroyed. Only the checkpoint remains.

── Phase 3: Recovering from checkpoint ──
  Available checkpoints: ['showcase-recovery-004', 'showcase-breaker

---

## Summary Dashboard

In [ ]:
print("=" * 80)
print(" HARDENING THE BACKBONE — SHOWCASE SUMMARY")
print("=" * 80)

# Gather all sessions we can find in checkpoints
all_sessions = []
for sid in list_checkpoints():
    s = load_checkpoint(sid)
    if s:
        all_sessions.append(s)

print(f"\n{'Session ID':<30} {'Status':<12} {'Turns':>6} {'Tokens':>8} {'Cost':>10} {'Checkpts':>10} {'Halt Reason'}")
print("─" * 110)
for s in sorted(all_sessions, key=lambda x: x.session_id):
    halt = s.halt_reason or "—"
    print(
        f"{s.session_id:<30} {s.status:<12} {s.turn_count:>6} "
        f"{s.total_tokens:>8} ${s.total_usd:>8.4f} {s.checkpoints_saved:>10}   {halt}"
    )

total_cost = sum(s.total_usd for s in all_sessions)
total_tokens = sum(s.total_tokens for s in all_sessions)
total_turns = sum(s.turn_count for s in all_sessions)

print(f"\n{'─' * 110}")
print(f"{'TOTAL':<30} {'':12} {total_turns:>6} {total_tokens:>8} ${total_cost:>8.4f}")

print(f"\n\n📋 Patterns Demonstrated:")
print(f"   ✅ Checkpointing  — {sum(s.checkpoints_saved for s in all_sessions)} checkpoints saved across all sessions")
print(f"   ✅ Context Pruning — Showcase 2 bloated history was trimmed before LLM call")
breaker_sessions = [s for s in all_sessions if s.status == "halted"]
print(f"   ✅ Circuit Breaker — {len(breaker_sessions)} session(s) halted by the breaker")
recovery_sessions = [s for s in all_sessions if "recovery" in s.session_id]
if recovery_sessions:
    print(f"   ✅ Crash Recovery  — Session '{recovery_sessions[0].session_id}' restored and continued")

 HARDENING THE BACKBONE — SHOWCASE SUMMARY

Session ID                     Status        Turns   Tokens       Cost   Checkpts Halt Reason
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
ckpt-demo                      running           3        0 $  0.0120          0   —
showcase-breaker-003           running           2     1310 $  0.0002          1   —
showcase-normal-001            completed         3     4055 $  0.0008          3   —
showcase-pruning-002           running           1     1403 $  0.0003          0   —
showcase-recovery-004          running           3     2062 $  0.0004          1   —

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
TOTAL                                           12     8830 $  0.0137


📋 Patterns Demonstrated:
   ✅ Checkpointing  — 5 checkpoints saved across all sessions
   ✅ Context Pruning — Showcase 2 bloated history was t

## Cleanup

In [ ]:
# Clean up the temporary checkpoint directory
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
print(f"Cleaned up checkpoint directory: {CHECKPOINT_DIR}")

Cleaned up checkpoint directory: /tmp/agent_checkpoints_qu63q9s0
